In [1]:
# Import core data analysis and machine learning libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid")

# Import models and metrics from scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
def eval_reg(y_true, y_pred, model_name="Model"):
    mae = mean_absolute_error(y_true, y_pred)
    # Using squared=False is deprecated in newer sklearn versions, so we use np.sqrt(mean_squared_error) for stability
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"--- {model_name} Performance ---")
    print(f"MAE:  {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²:   {r2:.4f}\n")

    return {"MAE": mae, "RMSE": rmse, "R2": r2}

In [4]:
import os
print(os.listdir("."))
# If there is a data folder in your current directory, check inside it:
# print(os.listdir("../data"))

['.config', 'student-merge.R', 'student-por.csv', 'student-mat.csv', 'student.txt', 'sample_data']


In [7]:
# 1. Mount Google Drive (if you are in Google Colab)
from google.colab import drive
drive.mount('/content/drive')

# 2. Update this path to match where your project folder is located in your Drive
# Example: df = pd.read_csv("/content/drive/MyDrive/student-performance-prediction-ml/student-mat.csv", sep=";")
df = pd.read_csv("student-mat.csv", sep=";")

# 3. Define target and features
target_col = "G3"
X = df.drop(columns=[target_col])
y = df[target_col]

# Convert categorical variables to numeric dummy variables
X = pd.get_dummies(X, drop_first=True)

# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Train and evaluate Extra Trees Regressor
et_model = ExtraTreesRegressor(random_state=42)
et_model.fit(X_train, y_train)
et_preds = et_model.predict(X_test)
et_metrics = eval_reg(y_test, et_preds, model_name="ExtraTrees Regressor")

# 5. Train and evaluate Gradient Boosting Regressor
gb_model = GradientBoostingRegressor(random_state=42)
gb_model.fit(X_train, y_train)
gb_preds = gb_model.predict(X_test)
gb_metrics = eval_reg(y_test, gb_preds, model_name="GradientBoosting Regressor")

Mounted at /content/drive
--- ExtraTrees Regressor Performance ---
MAE:  1.3377
RMSE: 2.2714
R²:   0.7484

--- GradientBoosting Regressor Performance ---
MAE:  1.1594
RMSE: 2.0047
R²:   0.8040



In [9]:
import pandas as pd

# 1. Initialize comparison_table if it doesn't exist yet from previous sessions
# (If you have previous model metrics, you can include them here)
comparison_table = pd.DataFrame(columns=["Model", "MAE", "RMSE", "R2"])

# 2. Set up your Session 29 results dataframe using your metrics
advanced_ensemble_results = pd.DataFrame([
    {"Model": "Extra Trees", "MAE": et_metrics["MAE"], "RMSE": et_metrics["RMSE"], "R2": et_metrics["R2"]},
    {"Model": "Gradient Boosting", "MAE": gb_metrics["MAE"], "RMSE": gb_metrics["RMSE"], "R2": gb_metrics["R2"]}
])

# 3. Standardize Session 29 names
advanced_ensemble_results["Model"] = advanced_ensemble_results["Model"].replace(
    {"ExtraTrees": "Extra Trees", "GradBoost": "Gradient Boosting"}
)

# 4. Add Session 29 models to the comparison table
comparison_table = pd.concat(
    [comparison_table, advanced_ensemble_results], ignore_index=True
)

# 5. Remove duplicates and rank by test RMSE
comparison_table = comparison_table.drop_duplicates(
    subset="Model", keep="last"
).sort_values(
    by="RMSE", ascending=True
).reset_index(drop=True)

comparison_table.insert(0, "Rank", range(1, len(comparison_table) + 1))

# 6. Create and display the final leaderboard
leaderboard = comparison_table[["Rank", "Model", "MAE", "RMSE", "R2"]].copy()
display(leaderboard.round(4))

# 7. Identify the current overall leader
current_best = leaderboard.iloc[0]
print("\nCurrent overall best model")
print("--------------------------")
print("Model:", current_best["Model"])
print(f"MAE:  {current_best['MAE']:.4f}")
print(f"RMSE: {current_best['RMSE']:.4f}")
print(f"R2:   {current_best['R2']:.4f}")

# 8. Identify the best Session 29 model
session29_leaderboard = leaderboard[
    leaderboard["Model"].isin(["Extra Trees", "Gradient Boosting"])
].sort_values(by="RMSE", ascending=True).reset_index(drop=True)

best_session29_model = session29_leaderboard.iloc[0]
print("\nBest Session 29 model")
print("---------------------")
print("Model:", best_session29_model["Model"])
print(f"RMSE:  {best_session29_model['RMSE']:.4f}")

print("\nStudent activity completed successfully.")

/tmp/ipykernel_450/2991166760.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  comparison_table = pd.concat(


,Rank,Model,MAE,RMSE,R2
0,1,Gradient Boosting,1.1594,2.0047,0.8040
1,2,Extra Trees,1.3377,2.2714,0.7484



Current overall best model
--------------------------
Model: Gradient Boosting
MAE:  1.1594
RMSE: 2.0047
R2:   0.8040

Best Session 29 model
---------------------
Model: Gradient Boosting
RMSE:  2.0047

Student activity completed successfully.


In [10]:
import pandas as pd

# Remove old rank column if present
comparison_table = comparison_table.drop(columns=["Rank"], errors="ignore")

# Standardize Session 29 names
advanced_ensemble_results = advanced_ensemble_results.copy()
advanced_ensemble_results["Model"] = advanced_ensemble_results["Model"].replace(
    {"ExtraTrees": "Extra Trees", "GradBoost": "Gradient Boosting"}
)

# Add Session 29 models
comparison_table = pd.concat(
    [comparison_table, advanced_ensemble_results], ignore_index=True
)

# Remove duplicates
comparison_table = comparison_table.drop_duplicates(
    subset="Model", keep="last"
)

# Rank by test RMSE
comparison_table = comparison_table.sort_values(
    by="RMSE", ascending=True
).reset_index(drop=True)
comparison_table.insert(0, "Rank", range(1, len(comparison_table) + 1))

# Create the final leaderboard
leaderboard = comparison_table[["Rank", "Model", "MAE", "RMSE", "R2"]].copy()
display(leaderboard.round(4))

# Identify the current overall leader
current_best = leaderboard.iloc[0]
print("\nCurrent overall best model")
print("--------------------------")
print("Model:", current_best["Model"])
print(f"MAE:  {current_best['MAE']:.4f}")
print(f"RMSE: {current_best['RMSE']:.4f}")
print(f"R2:   {current_best['R2']:.4f}")

# Identify the best Session 29 model
session29_leaderboard = leaderboard[
    leaderboard["Model"].isin(["Extra Trees", "Gradient Boosting"])
].sort_values(by="RMSE", ascending=True).reset_index(drop=True)

best_session29_model = session29_leaderboard.iloc[0]
print("\nBest Session 29 model")
print("---------------------")
print("Model:", best_session29_model["Model"])
print(f"RMSE:  {best_session29_model['RMSE']:.4f}")

print("\nStudent activity completed successfully.")

,Rank,Model,MAE,RMSE,R2
0,1,Gradient Boosting,1.1594,2.0047,0.8040
1,2,Extra Trees,1.3377,2.2714,0.7484



Current overall best model
--------------------------
Model: Gradient Boosting
MAE:  1.1594
RMSE: 2.0047
R2:   0.8040

Best Session 29 model
---------------------
Model: Gradient Boosting
RMSE:  2.0047

Student activity completed successfully.
